In [1]:
import pandas as pd
from datetime import datetime



 # code from forecasting
 ```
apcd_s3_output_path='tijuana/sd_apcd_air/output'
    h2surl = s3_resource.publicUrl(path=f'{apcd_s3_output_path}/h2s.csv', bucket=s3_resource.S3_BUCKET)
```

In [9]:
h2surl = 'https://oss.resilientservice.mooo.com/resilentpublic/tijuana/sd_apcd_air/output/h2s.csv'
forecast_url = 'https://oss.resilientservice.mooo.com/resilentpublic/tijuana/weather/raw/forecast.csv'
w2025_url = 'https://oss.resilientservice.mooo.com/resilentpublic/tijuana/weather/raw/2025.csv'
w2024_url = 'https://oss.resilientservice.mooo.com/rresilentpublic/tijuana/weather/raw/2024.csv'
w2023_url = 'https://oss.resilientservice.mooo.com/resilentpublic/tijuana/weather/raw/2023.csv'
streamflow_border_url = 'https://oss.resilientservice.mooo.com/resilentpublic/tijuana/streamflow/output/boundary_cms.csv'
complaints_url='https://oss.resilientservice.mooo.com/resilentpublic/tijuana/sd_complaints/output/complaints_by_date.csv'

In [51]:
# clean h2s
h2s_sensor_data_all = pd.read_csv(h2surl)
# using names causes a parsing error, do just drop after loading
h2s_sensor_data_all = h2s_sensor_data_all.drop(['Original Value', 'Icons', 'level', 'Parameter', 'LongName', 'Site Name', 'Latitude',	'Longitude',	'AgencyName',	'geometry' ], axis=1)
h2s_sensor_data_all['time'] = pd.to_datetime(h2s_sensor_data_all['Date with time'], utc=True)
h2s_sensor_data_all =  h2s_sensor_data_all.rename(columns={'Result': 'H2S', 'Qualifier':  'H2S_qualifier'})

#2s_sensor_data_all.index = pd.to_datetime(h2s_sensor_data_all['Date with time']).dt.tz_localize('America/Los_Angeles', ambiguous=True)
h2s_sensor_data_all = h2s_sensor_data_all.drop('Date with time', axis=1)
h2s_sensor_data_all = h2s_sensor_data_all.set_index(pd.DatetimeIndex(h2s_sensor_data_all['time']))
h2s_sensor_data_all = h2s_sensor_data_all.drop('time', axis=1)
h2s_sensor_data_all = h2s_sensor_data_all.sort_index()
h2s_sensor_data_all.head()

,SiteName,H2S,H2S_qualifier
time,,,
2025-08-01 08:00:00+00:00,NESTOR - BES,2.1,NaN
2025-08-01 08:00:00+00:00,SAN YSIDRO,NaN,NaN
2025-08-01 08:00:00+00:00,IB CIVIC CTR,NaN,NaN
2025-08-01 09:00:00+00:00,SAN YSIDRO,NaN,NaN
2025-08-01 09:00:00+00:00,NESTOR - BES,1.9,NaN


In [32]:
## forecast
forecast_df = pd.read_csv(forecast_url)
forecast_df['time'] = pd.to_datetime(forecast_df['date'], utc=True)
#forecast_df["time"] = forecast_df["time"].dt.tz_localize("America/Los_Angeles", ambiguous=True)
forecast_df = forecast_df.set_index(pd.DatetimeIndex(forecast_df['time']))
forecast_df = forecast_df.drop(['time','date'], axis=1)
forecast_df.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m,precipitation,surface_pressure,cloud_cover,visibility,dewpoint_2m
time,,,,,,,,,
2025-10-21 00:00:00+00:00,22.2885,9.826088,298.44280,61.0,0.0,1013.02435,5.0,28800.0,14.406236
2025-10-21 01:00:00+00:00,19.9385,7.704336,307.40543,71.0,0.0,1013.20900,8.0,22400.0,14.522919
2025-10-21 02:00:00+00:00,18.4385,3.758510,343.30070,74.0,0.0,1014.09740,5.0,21300.0,13.719246
2025-10-21 03:00:00+00:00,16.8385,4.802999,347.00537,87.0,0.0,1014.68580,20.0,15100.0,14.661227
2025-10-21 04:00:00+00:00,16.8385,1.138420,108.43504,89.0,0.0,1015.08510,50.0,14100.0,15.014073


In [33]:
# historical
weather_df = pd.read_csv(w2025_url)
weather_df['time'] = pd.to_datetime(weather_df['date'], utc=True)
#forecast_df["time"] = forecast_df["time"].dt.tz_localize("America/Los_Angeles", ambiguous=True)
weather_df = weather_df.set_index(pd.DatetimeIndex(weather_df['time']))
weather_df = weather_df.drop(['time','date'], axis=1)

weather_df.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,precipitation,relative_humidity_2m,surface_pressure,cloud_cover,visibility,dewpoint_2m
time,,,,,,,,,,
2025-01-01 00:00:00+00:00,0.8085,19.036650,220.01361,40.680000,0.0,83.363240,1014.18490,100.0,NaN,-1.6915
2025-01-01 01:00:00+00:00,2.1085,21.087675,219.80553,44.639996,0.0,83.221330,1013.11300,100.0,NaN,-0.4415
2025-01-01 02:00:00+00:00,2.5585,21.809732,221.31927,46.800000,0.0,82.376150,1012.62286,100.0,NaN,-0.1415
2025-01-01 03:00:00+00:00,3.1085,21.905340,223.00212,46.800000,0.0,81.558304,1011.73670,100.0,NaN,0.2585
2025-01-01 04:00:00+00:00,3.6085,22.277134,225.98203,47.519997,0.0,80.747880,1011.14810,100.0,NaN,0.6085


In [52]:
matched_df = pd.merge_asof(h2s_sensor_data_all, weather_df, left_on="time", right_on="time", direction="nearest")
matched_df.head()

,time,SiteName,H2S,H2S_qualifier,temperature_2m,wind_speed_10m,wind_direction_10m,wind_gusts_10m,precipitation,relative_humidity_2m,surface_pressure,cloud_cover,visibility,dewpoint_2m
0,2025-08-01 08:00:00+00:00,NESTOR - BES,2.1,NaN,18.108500,9.144637,259.79608,23.039999,0.0,77.73137,1004.21594,92.0,NaN,14.158501
1,2025-08-01 08:00:00+00:00,SAN YSIDRO,NaN,NaN,18.108500,9.144637,259.79608,23.039999,0.0,77.73137,1004.21594,92.0,NaN,14.158501
2,2025-08-01 08:00:00+00:00,IB CIVIC CTR,NaN,NaN,18.108500,9.144637,259.79608,23.039999,0.0,77.73137,1004.21594,92.0,NaN,14.158501
3,2025-08-01 09:00:00+00:00,SAN YSIDRO,NaN,NaN,19.008501,10.018743,252.21602,27.359999,0.0,71.12814,1004.22980,81.0,NaN,13.658501
4,2025-08-01 09:00:00+00:00,NESTOR - BES,1.9,NaN,19.008501,10.018743,252.21602,27.359999,0.0,71.12814,1004.22980,81.0,NaN,13.658501


In [54]:
# streamflow
stream_border_df = pd.read_csv(streamflow_border_url)
stream_border_df.head()

,End of Interval (UTC-08:00),Average (m^3/s)
0,2025-10-21 01:00:00,1.70
1,2025-10-21 02:00:00,1.71
2,2025-10-21 03:00:00,1.69
3,2025-10-21 04:00:00,1.55
4,2025-10-21 05:00:00,1.48


In [55]:
stream_border_df['time'] = pd.to_datetime(stream_border_df['End of Interval (UTC-08:00)'], utc=False)
stream_border_df["time"] = stream_border_df["time"].dt.tz_localize("America/Los_Angeles", ambiguous=True)
stream_border_df = stream_border_df.rename(columns={'Average (m^3/s)': 'Flow (m^3/s)--Border'})
stream_border_df = stream_border_df.set_index(pd.DatetimeIndex(stream_border_df['time']))
stream_border_df = stream_border_df.drop(['time','End of Interval (UTC-08:00)'], axis=1)

stream_border_df.head()


,Flow (m^3/s)--Border
time,
2025-10-21 01:00:00-07:00,1.70
2025-10-21 02:00:00-07:00,1.71
2025-10-21 03:00:00-07:00,1.69
2025-10-21 04:00:00-07:00,1.55
2025-10-21 05:00:00-07:00,1.48
